# Multiple Linear Regression - An Example

Suppose that the sales department of some business wants to study how a product total sales is influenced by publicity on different channels. We have a database with revenue data (in millions of dollars) obtained from sales in 200 regions, as well as the budget (also in millions of dollars) for ads in radio, TV, and newspapers in each of those regions.

## Libraries

These are the libraries used in this analysis, together with some setup options:

In [ ]:
# Data wrangling
# ==============================================================================
import pandas as pd
import numpy as np

In [ ]:
# Plots
# ==============================================================================
import matplotlib.pyplot as plt
#from matplotlib import style
import seaborn as sns

In [ ]:
# Preprocessing and modeling
# ==============================================================================
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from scipy import stats

In [ ]:
# Configuring matplotlib and seaborn
# ==========================================================================bb====
from matplotlib.colors import ListedColormap
plt.rcParams['savefig.bbox'] = "tight"
sns.set_style('whitegrid') # or plt.style.use('ggplot')
sns.set_palette('viridis')

In [ ]:
# For a list of colormaps uncomment the following lines
#from matplotlib import colormaps
#sorted(list(colormaps))

In [ ]:
# Configuring warnings
# ==============================================================================
# If you want to suppress warnings uncomment the following lines
#import warnings
#warnings.filterwarnings('ignore')

## Data

In [ ]:
tv = [230.1, 44.5, 17.2, 151.5, 180.8, 8.7, 57.5, 120.2, 8.6, 199.8, 66.1, 214.7,
      23.8, 97.5, 204.1, 195.4, 67.8, 281.4, 69.2, 147.3, 218.4, 237.4, 13.2,
      228.3, 62.3, 262.9, 142.9, 240.1, 248.8, 70.6, 292.9, 112.9, 97.2, 265.6,
      95.7, 290.7, 266.9, 74.7, 43.1, 228.0, 202.5, 177.0, 293.6, 206.9, 25.1,
      175.1, 89.7, 239.9, 227.2, 66.9, 199.8, 100.4, 216.4, 182.6, 262.7, 198.9,
      7.3, 136.2, 210.8, 210.7, 53.5, 261.3, 239.3, 102.7, 131.1, 69.0, 31.5,
      139.3, 237.4, 216.8, 199.1, 109.8, 26.8, 129.4, 213.4, 16.9, 27.5, 120.5,
      5.4, 116.0, 76.4, 239.8, 75.3, 68.4, 213.5, 193.2, 76.3, 110.7, 88.3, 109.8,
      134.3, 28.6, 217.7, 250.9, 107.4, 163.3, 197.6, 184.9, 289.7, 135.2, 222.4,
      296.4, 280.2, 187.9, 238.2, 137.9, 25.0, 90.4, 13.1, 255.4, 225.8, 241.7, 175.7,
      209.6, 78.2, 75.1, 139.2, 76.4, 125.7, 19.4, 141.3, 18.8, 224.0, 123.1, 229.5,
      87.2, 7.8, 80.2, 220.3, 59.6, 0.7, 265.2, 8.4, 219.8, 36.9, 48.3, 25.6, 273.7,
      43.0, 184.9, 73.4, 193.7, 220.5, 104.6, 96.2, 140.3, 240.1, 243.2, 38.0, 44.7,
      280.7, 121.0, 197.6, 171.3, 187.8, 4.1, 93.9, 149.8, 11.7, 131.7, 172.5, 85.7,
      188.4, 163.5, 117.2, 234.5, 17.9, 206.8, 215.4, 284.3, 50.0, 164.5, 19.6, 168.4,
      222.4, 276.9, 248.4, 170.2, 276.7, 165.6, 156.6, 218.5, 56.2, 287.6, 253.8, 205.0,
      139.5, 191.1, 286.0, 18.7, 39.5, 75.5, 17.2, 166.8, 149.7, 38.2, 94.2, 177.0,
      283.6, 232.1]

radio = [37.8, 39.3, 45.9, 41.3, 10.8, 48.9, 32.8, 19.6, 2.1, 2.6, 5.8, 24.0, 35.1,
         7.6, 32.9, 47.7, 36.6, 39.6, 20.5, 23.9, 27.7, 5.1, 15.9, 16.9, 12.6, 3.5,
         29.3, 16.7, 27.1, 16.0, 28.3, 17.4, 1.5, 20.0, 1.4, 4.1, 43.8, 49.4, 26.7,
         37.7, 22.3, 33.4, 27.7, 8.4, 25.7, 22.5, 9.9, 41.5, 15.8, 11.7, 3.1, 9.6,
         41.7, 46.2, 28.8, 49.4, 28.1, 19.2, 49.6, 29.5, 2.0, 42.7, 15.5, 29.6, 42.8,
         9.3, 24.6, 14.5, 27.5, 43.9, 30.6, 14.3, 33.0, 5.7, 24.6, 43.7, 1.6, 28.5,
         29.9, 7.7, 26.7, 4.1, 20.3, 44.5, 43.0, 18.4, 27.5, 40.6, 25.5, 47.8, 4.9,
         1.5, 33.5, 36.5, 14.0, 31.6, 3.5, 21.0, 42.3, 41.7, 4.3, 36.3, 10.1, 17.2,
         34.3, 46.4, 11.0, 0.3, 0.4, 26.9, 8.2, 38.0, 15.4, 20.6, 46.8, 35.0, 14.3,
         0.8, 36.9, 16.0, 26.8, 21.7, 2.4, 34.6, 32.3, 11.8, 38.9, 0.0, 49.0, 12.0,
         39.6, 2.9, 27.2, 33.5, 38.6, 47.0, 39.0, 28.9, 25.9, 43.9, 17.0, 35.4, 33.2,
         5.7, 14.8, 1.9, 7.3, 49.0, 40.3, 25.8, 13.9, 8.4, 23.3, 39.7, 21.1, 11.6, 43.5,
         1.3, 36.9, 18.4, 18.1, 35.8, 18.1, 36.8, 14.7, 3.4, 37.6, 5.2, 23.6, 10.6, 11.6,
         20.9, 20.1, 7.1, 3.4, 48.9, 30.2, 7.8, 2.3, 10.0, 2.6, 5.4, 5.7, 43.0, 21.3, 45.1,
         2.1, 28.7, 13.9, 12.1, 41.1, 10.8, 4.1, 42.0, 35.6, 3.7, 4.9, 9.3, 42.0, 8.6]

newspaper = [69.2, 45.1, 69.3, 58.5, 58.4, 75.0, 23.5, 11.6, 1.0, 21.2, 24.2, 4.0,
             65.9, 7.2, 46.0, 52.9, 114.0, 55.8, 18.3, 19.1, 53.4, 23.5, 49.6, 26.2,
             18.3, 19.5, 12.6, 22.9, 22.9, 40.8, 43.2, 38.6, 30.0, 0.3, 7.4, 8.5, 5.0,
             45.7, 35.1, 32.0, 31.6, 38.7, 1.8, 26.4, 43.3, 31.5, 35.7, 18.5, 49.9,
             36.8, 34.6, 3.6, 39.6, 58.7, 15.9, 60.0, 41.4, 16.6, 37.7, 9.3, 21.4, 54.7,
             27.3, 8.4, 28.9, 0.9, 2.2, 10.2, 11.0, 27.2, 38.7, 31.7, 19.3, 31.3, 13.1,
             89.4, 20.7, 14.2, 9.4, 23.1, 22.3, 36.9, 32.5, 35.6, 33.8, 65.7, 16.0, 63.2,
             73.4, 51.4, 9.3, 33.0, 59.0, 72.3, 10.9, 52.9, 5.9, 22.0, 51.2, 45.9, 49.8,
             100.9, 21.4, 17.9, 5.3, 59.0, 29.7, 23.2, 25.6, 5.5, 56.5, 23.2, 2.4, 10.7,
             34.5, 52.7, 25.6, 14.8, 79.2, 22.3, 46.2, 50.4, 15.6, 12.4, 74.2, 25.9, 50.6,
             9.2, 3.2, 43.1, 8.7, 43.0, 2.1, 45.1, 65.6, 8.5, 9.3, 59.7, 20.5, 1.7, 12.9,
             75.6, 37.9, 34.4, 38.9, 9.0, 8.7, 44.3, 11.9, 20.6, 37.0, 48.7, 14.2, 37.7,
             9.5, 5.7, 50.5, 24.3, 45.2, 34.6, 30.7, 49.3, 25.6, 7.4, 5.4, 84.8, 21.6, 19.4,
             57.6, 6.4, 18.4, 47.4, 17.0, 12.8, 13.1, 41.8, 20.3, 35.2, 23.7, 17.6, 8.3,
             27.4, 29.7, 71.8, 30.0, 19.6, 26.6, 18.2, 3.7, 23.4, 5.8, 6.0, 31.6, 3.6, 6.0,
             13.8, 8.1, 6.4, 66.2, 8.7]

sales = [22.1, 10.4, 9.3, 18.5, 12.9, 7.2, 11.8, 13.2, 4.8, 10.6, 8.6, 17.4, 9.2, 9.7,
          19.0, 22.4, 12.5, 24.4, 11.3, 14.6, 18.0, 12.5, 5.6, 15.5, 9.7, 12.0, 15.0, 15.9,
          18.9, 10.5, 21.4, 11.9, 9.6, 17.4, 9.5, 12.8, 25.4, 14.7, 10.1, 21.5, 16.6, 17.1,
          20.7, 12.9, 8.5, 14.9, 10.6, 23.2, 14.8, 9.7, 11.4, 10.7, 22.6, 21.2, 20.2, 23.7,
          5.5, 13.2, 23.8, 18.4, 8.1, 24.2, 15.7, 14.0, 18.0, 9.3, 9.5, 13.4, 18.9, 22.3,
          18.3, 12.4, 8.8, 11.0, 17.0, 8.7, 6.9, 14.2, 5.3, 11.0, 11.8, 12.3, 11.3, 13.6,
          21.7, 15.2, 12.0, 16.0, 12.9, 16.7, 11.2, 7.3, 19.4, 22.2, 11.5, 16.9, 11.7, 15.5,
          25.4, 17.2, 11.7, 23.8, 14.8, 14.7, 20.7, 19.2, 7.2, 8.7, 5.3, 19.8, 13.4, 21.8,
          14.1, 15.9, 14.6, 12.6, 12.2, 9.4, 15.9, 6.6, 15.5, 7.0, 11.6, 15.2, 19.7, 10.6,
          6.6, 8.8, 24.7, 9.7, 1.6, 12.7, 5.7, 19.6, 10.8, 11.6, 9.5, 20.8, 9.6, 20.7, 10.9,
          19.2, 20.1, 10.4, 11.4, 10.3, 13.2, 25.4, 10.9, 10.1, 16.1, 11.6, 16.6, 19.0, 15.6,
          3.2, 15.3, 10.1, 7.3, 12.9, 14.4, 13.3, 14.9, 18.0, 11.9, 11.9, 8.0, 12.2, 17.1,
          15.0, 8.4, 14.5, 7.6, 11.7, 11.5, 27.0, 20.2, 11.7, 11.8, 12.6, 10.5, 12.2, 8.7,
          26.2, 17.6, 22.6, 10.3, 17.3, 15.9, 6.7, 10.8, 9.9, 5.9, 19.6, 17.3, 7.6, 9.7, 12.8,
          25.5, 13.4]

data = pd.DataFrame({'tv': tv, 'radio': radio, 'newspaper':newspaper, 'sales': sales})
mean_predicted_variable = data['sales'].mean()


In [ ]:
data.head()

## Relations between variables

The first step in developing a multiple linear model is to study the relationship that exists between the different variables. This information is very useful when identifying which are the best predictors for the model, and for detecting collinearity between predictors.

In [ ]:
# Correlation between numerical columns
# ==============================================================================

def tidy_corr_matrix(corr_mat):
    '''
    This functions converts a pandas correlation matrix to tidy format
    '''
    corr_mat = corr_mat.stack().reset_index()
    corr_mat.columns = ['variable_1','variable_2','r']
    corr_mat = corr_mat.loc[corr_mat['variable_1'] != corr_mat['variable_2'], :]
    corr_mat['abs_r'] = np.abs(corr_mat['r'])
    corr_mat = corr_mat.sort_values('abs_r', ascending=False)
    
    return(corr_mat)



corr_matrix = data.select_dtypes(include=['float64', 'int']).corr(method='pearson')
tidy_corr_matrix(corr_matrix).head(24)

In [ ]:
# Heatmap for correlation matrix
# ==============================================================================
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))

sns.heatmap(
    corr_matrix,
    annot     = True,
    cbar      = False,
    annot_kws = {"size": 8},
    vmin      = -1,
    vmax      = 1,
    center    = 0,
    #cmap      = sns.light_palette('purple', n_colors = 200),
    cmap      = sns.diverging_palette(20, 200, n=200),
    square    = True,
    ax        = ax
)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation = 45,
    horizontalalignment = 'right',
)

ax.tick_params(labelsize = 10)

We observe that `tv` has the highest correlation with `sales`, while `newspaper` has the smallest such correlation. This indicates that perhaps, after fitting our model, we can eliminate `newspaper` from the model and get a better prediction.

In [ ]:
# Distribution plots for each numerical variable
# ==============================================================================
# 
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(9, 5))
axes = axes.flat
columnas_numeric = data.select_dtypes(include=['float64', 'int']).columns

for i, colum in enumerate(columnas_numeric):
    sns.histplot(
        data    = data,
        x       = colum,
        stat    = 'count',
        kde     = True,
        color   = (list(plt.rcParams['axes.prop_cycle'])*2)[i]["color"],
        line_kws= {'linewidth': 2},
        alpha   = 0.3,
        ax      = axes[i]
    )
    axes[i].set_title(colum, fontsize = 10, fontweight = "bold")
    axes[i].tick_params(labelsize = 8)
    axes[i].set_xlabel("")


    
fig.tight_layout()
plt.subplots_adjust(top = 0.9)
fig.suptitle('Distributions', fontsize = 10, fontweight = "bold");

In [ ]:
data.describe()

## Fitting the model

We fit a multiple linear regression model with the objective of predicting sales in function of the budget assigned for the three ad channels.

We will use **Statsmodels** for fitting the linear regression model, presenting all variants of the process as we did with simple linear regression.

### Setting up the data for training and testing the model

In [ ]:
# Train-test Splitting
# ==============================================================================
X = data[['tv', 'radio', 'newspaper']]
y = data['sales']

X_train, X_test, y_train, y_test = train_test_split(
                                        X,
                                        y.values.reshape(-1,1),
                                        train_size   = 0.8,
                                        random_state = 1234,
                                        shuffle      = True
                                    )

In [ ]:
data_train = pd.DataFrame(
                   np.hstack((X_train, y_train)),
                   columns=['tv', 'radio', 'newspaper', 'sales']
              )

In [ ]:
data_train

### Model fitting with a formula

In [ ]:
# Model fitting using a formula
# ==============================================================================
modelf = smf.ols(formula = 'sales ~ tv + radio + newspaper', data = data_train)
modelf = modelf.fit()
modelf.summary()

In [ ]:
modelf_noint = smf.ols(formula = 'sales ~ tv + radio + newspaper + 0', data = data_train)
modelf_noint = modelf_noint.fit()
modelf_noint.summary()

### Model fitting with matrices

In [ ]:
# Model fitting using matrices
# ==============================================================================
# Add a column of 1s to the predictor matrix
X_train_int = sm.add_constant(X_train, prepend=True)

In [ ]:
modelm = sm.OLS(endog=y_train, exog=X_train_int)
modelm = modelm.fit()
modelm.summary()

In [ ]:
modelm_noint = sm.OLS(endog=y_train, exog=X_train)
modelm_noint = modelm_noint.fit()
modelm_noint.summary()

Our model has a high $R^2$ value (0.894), which means it can explain 89.4% of the observed sales variability. The *$p$-value* for the model is highly significant (1.01e-75), so we can accept the hypothesis that our model is better than what would be expected of random chance.

### Eliminating variables that have no significant predictive power

We observe that the *$p$-value* for `newspaper` is high (0.723), which means that this variable does not significantly contributes to our model, just as we explained with the correlation matrix. We can get rid off this variable, and train the new model.

In [ ]:
# Model fitting using matrices
# ==============================================================================
# Drop `newspaper`
X_train_new = X_train.drop(columns = 'newspaper')
X_test_new  = X_test.drop(columns = 'newspaper')

# Add a column of 1s to the predictor matrix
X_train_int_new = sm.add_constant(X_train_new, prepend=True)
modelm_new  = sm.OLS(endog=y_train, exog=X_train_int_new)
modelm_new  = modelm_new.fit()
modelm_new.summary()

In [ ]:
# Model fitting using a formula
# ==============================================================================
model = smf.ols(formula = 'sales ~ tv + radio', data = data_train)
model = model.fit()
model.summary()

## Confidence intervals for the coefficients

In [ ]:
# Confidence intervals for the coefficients
# ==============================================================================
intervals_ci = model.conf_int(alpha=0.05)
intervals_ci.columns = ['2.5%', '97.5%']
intervals_ci

## Residual analysis

In [ ]:
# Residual analysis
# ==============================================================================
y_train = y_train.flatten()
prediction_train = model.predict(exog = X_train_int)
residues_train   = prediction_train - y_train

In [ ]:
print(y_train.shape, prediction_train.shape, residues_train.shape)

### Graphs

In [ ]:
# Graphs
# ==============================================================================
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(9, 8))

# axes[0,0]
sns.scatterplot(
    x          = y_train,
    y          = prediction_train,
    edgecolors = (0, 0, 0),
    alpha      = 0.4,
    ax         = axes[0, 0]
)

points = [(y_train.min(),prediction_train.min()), (y_train.max(), prediction_train.max())]  # x, y coordinates
line_data = pd.DataFrame(points, columns=['x', 'y'])
sns.lineplot(
    x          = 'x',
    y          = 'y',
    data       = line_data, 
    linestyle  = '--', 
    color      = 'gray', 
    lw         = 2,
    ax         = axes[0, 0]
)

axes[0, 0].set_title('Predicted value vs real value', fontsize = 10, fontweight = "bold")
axes[0, 0].set_xlabel('Real')
axes[0, 0].set_ylabel('Prediction')
axes[0, 0].tick_params(labelsize = 7)

# axes[0,1]
sns.scatterplot(
    x          = list(range(len(y_train))),
    y          = residues_train,
    edgecolors = (0, 0, 0),
    alpha      = 0.4,
    ax         = axes[0, 1]
)

axes[0, 1].axhline(y = 0, linestyle = '--', color = 'gray', lw=2)
axes[0, 1].set_title('Residues', fontsize = 10, fontweight = "bold")
axes[0, 1].set_xlabel('id')
axes[0, 1].set_ylabel('Residue')
axes[0, 1].tick_params(labelsize = 7)

# axes[1, 0]
sns.histplot(
    data    = residues_train,
    stat    = "density",
    kde     = True,
    line_kws= {'linewidth': 1},
    alpha   = 0.3,
    ax      = axes[1, 0]
)

axes[1, 0].set_title('Distribution of residues', fontsize = 10,
                     fontweight = "bold")
axes[1, 0].set_xlabel("Residue")
axes[1, 0].tick_params(labelsize = 7)

# axes[1, 1]
sm.qqplot(
    residues_train,
    fit   = True,
    line  = 'q',
    ax    = axes[1, 1], 
    alpha = 0.4,
    lw    = 2
)

axes[1, 1].set_title('Q-Q residues', fontsize = 10, fontweight = "bold")
axes[1, 1].tick_params(labelsize = 7)

# axes[2, 0]
sns.scatterplot(
    x          = prediction_train, 
    y          = residues_train,
    edgecolors = (0, 0, 0),
    alpha      = 0.4,
    ax         = axes[2, 0]
)

axes[2, 0].axhline(y = 0, linestyle = '--', color = 'grey', lw=2)
axes[2, 0].set_title('Residues vs prediction', fontsize = 10, fontweight = "bold")
axes[2, 0].set_xlabel('Prediction')
axes[2, 0].set_ylabel('Residue')
axes[2, 0].tick_params(labelsize = 7)

# Eliminate empty axes
fig.delaxes(axes[2,1])

fig.tight_layout()
plt.subplots_adjust(top=0.9)
fig.suptitle('Residual analysis', fontsize = 12, fontweight = "bold");

Residues do not appear to randomly distribute around zero without having the same variability along the X axis. This pattern hints to a lack of homoscedasticity.

### Normality test

We now verify if the residues follow a normal distribution using to statistical tests: *Shapiro-Wilk test* and *D'Agostino's K-squared test*. The latter includes a summary from **Statsmodels** with the name *Omnibus*.

On both tests, the null hypothesis considers that the data follows a normal distribution. Hence, if the *$p$-value* is not smaller than the selected significance level *alpha* there is no evidence to reject the statement that the data is normally distributed.

In [ ]:
# Shapiro-Wilk test
# ==============================================================================
shapiro_test = stats.shapiro(residues_train)
shapiro_test

In [ ]:
# D'Agostino's K-squared test
# ==============================================================================
k2, p_value = stats.normaltest(residues_train)
print(f"Statistic= {k2}, p-value = {p_value}")

Both tests show clear evidence for rejecting the hypothesis that data is normally distributed (*$p$-value* << 0.01).

## Predictions

Now that we have trained our model, we can obtain predictions for new data. Models from **statsmodels** allow us to calculate confidence intervals associated with each prediction.

In [ ]:
# # Predictions with 95% confidence intervals
# ==============================================================================
predictions = model.get_prediction(exog = X_train_int).summary_frame(alpha=0.05)
predictions.head(4)

## Model error test

In [ ]:
# Model error test
# ==============================================================================
X_test_int = sm.add_constant(X_test, prepend=True)
predictions = model.predict(exog = X_test_int)
rmse = root_mean_squared_error(
        y_true  = y_test,
        y_pred  = predictions,
       )
print("")
print(f"The test error (rmse) is {rmse}, which is {rmse/mean_predicted_variable*100:.2f}% of the mean of `sales`.")

In [ ]:
residues=y_test[:,0]-predictions
residues.sum()

In [ ]:
model.params

## Interpretating our results

The multiple linear regression that we obtained,

$$sales = 2.9004 + 0.0456tv + 0.1904radio,$$

can explain 89.4% of the observed sales variance (*R-squared*: 0.894, *Adj. R-squared*: 0.892). The *F* test is significant (*$p$-value*: 3.69e-77). Therefore, there is clear evidence that the variance explained by the model is greater than that of random chance.

Normality is not satisfied, so the estimated confidence intervals for the coefficients are not reliable.

The model error test is 1.696, so the model predictions get farther from the real value by 1.696 **on average**.

## Interaction between predictors

The linear model from which the conclusions have been drawn assumes that the effect on sales due to an increase in the budget of one of the media is independent of the budget spent on the others. For example, the linear model assumes that the average effect on sales of increasing the TV ad budget by one unit is always 0.0456, regardless of the amount spent on radio ads. However, this does not necessarily have to be so, there may be interaction between the predictors in such a way that the effect of each of them on the response variable depends to a certain extent on the value taken by the other predictor.

Recall that a two-predictor linear model follows the equation

$$y=\beta_0+\beta_1 x_1+\beta_2 x_2+\varepsilon$$
 
According to this definition, an increase of one unit of the predictor $x_1$ produces an average increment of $\beta_1$ on the variable $y$. Modifying the predictor $x_2$ does not change this, and the same happens with $x_2$ with respect to $x_1$. If we want our model to include the interaction between predictors, we introduce a third predictor, the *interaction term*, which is the product of the predictors $x_1$ y $x_2$. Thus, our proposed model is

$$y=\beta_0 + \beta_1 x_1 + \beta_2 x_2 + \beta_3 x_1 x_2 + \varepsilon.$$
 
or equivalently,

$$y=\beta_0 + (\beta_1 + \beta_3 x_2) x_1 + \beta_2 x_2 + \varepsilon.$$
 
The effect of $x_1$ over $y$ is not constant anymore, but depends on the value of $x_2$.

For those multiple linear regression models that include interactions between predictors we have to take into account the *hierarchical principle*, according to which if we include interaction between predictors, we must include the individual predictors as well, regardless of whether their *$p$-value* is significant or not.

With **Statsmodels** we may introduce the interaction between predictors in two ways:

- Adding the interaction term to our formula: `sales ~ tv * radio`

- Adding a new column with the product of the predictors for which the interaction need to be considered.

In [ ]:
# Train-test split
# ==============================================================================
X = data[['tv', 'radio']]
y = data['sales']

X_train, X_test, y_train, y_test = train_test_split(
                                        X,
                                        y.values.reshape(-1,1),
                                        train_size   = 0.8,
                                        random_state = 1234,
                                        shuffle      = True
                                    )

In [ ]:
# Creating the model with interaction included in the formula (similar to R)
# ==============================================================================
data = pd.DataFrame(
                     np.hstack((X_train, y_train)),
                     columns=['tv', 'radio', 'sales']
)

In [ ]:
# Creating the model with interaction included in the formula (similar to R)
# ==============================================================================
# 
model_interaction = smf.ols(formula = 'sales ~ tv*radio', data = data)
model_interaction = model_interaction.fit()
model_interaction.summary()

In [ ]:
# Creating the model with interactions using matrices
# ==============================================================================
# Adding a new column with the desired interaction
X_train_new = X_train
X_train_new['tv_radio'] = X_train_new['tv'] * X_train_new['radio']
X_test_new['tv_radio']  = X_test_new['tv'] * X_test_new['radio']

# Adding a column of 1s for the intercept
X_train_new = sm.add_constant(X_train_new, prepend=True)
model_interaction_m  = sm.OLS(endog=y_train, exog=X_train_new)
model_interaction_m  = model_interaction_m.fit()
model_interaction_m.summary()

Results show clear evidence that the interaction _tv*radio_ is significant (*$p$-value* very close to 0).

The model with interactions has an *Adjusted R-squared* of 0.967, which is higher than that of the model without interactions (0.892). Is this difference sufficient to conclude that the new model is superior to the old one? We use an $F$-test (ANOVA) to answer this question.

## Model comparison using an $F$-test (ANOVA)

Consider two models $M$ and $m$, where the latter is formed with a subset of the predictors of the former. If the difference of the fittings is small, the smaller model is more adeacuate. It is possible to determine if the difference in fittings is significant by comparing the residues. We use the statistic

$$\frac{RSS_m−RSS_M}{RSS_M}$$
 
In order to eliminate the influence of the size of the model, we divide the residual sum of squares $RSS$ of each model by their degrees of freedom. Thus, the new statistic follows an $F$ distribution

$$\frac{(RSS_m−RSS_M)/(df_m−df_M)}{RSS_M/(df_M)}\sim F_{df_m−df_M, df_M}$$,

where $df$ denotes the degrees of freedom of the model, which is the number of observations minus the number of predictors.

With **Statsmodels** we determine if the new model better fits the data than the old model with the function `anova_lm()` from `statsmodels.stats.anova`.

In [ ]:
anova_lm(model, model_interaction)

The *ANOVA* test shows clear evidence (*$p$-value* is basically zero) that our model with interactions between predictors explains the response variable better than that without interactions.

## References

Regresión lineal con Python by Joaquín Amat Rodrigo, available under a Attribution 4.0 International (CC BY 4.0) at https://www.cienciadedatos.net/documentos/py10-regresion-lineal-python.html